### Step 1: Install necesscary packages

In [46]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [47]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5

#device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.backends.mps.is_available(): #changed this cause using MacBook, otherwise use above
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [48]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [49]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [50]:
# Load data from ./data/pos_neg_pairs.json
data_path = "pos_neg_pairs_generated.json"#for now im using _generated, same name as my script

with open(data_path, "r", encoding="utf-8") as f:
    lines = json.load(f)

# Validate dataset
assert isinstance(lines, list), "Dataset should be a list."
assert len(lines) > 0, "Dataset is empty."

for i, pair in enumerate(lines):
    assert isinstance(pair, dict), f"Item {i} is not a dictionary."
    assert "negative" in pair, f"Item {i} is missing 'negative'."
    assert "positive" in pair, f"Item {i} is missing 'positive'."

print("Dataset loaded successfully.")
print("Number of preference pairs:", len(lines))

# Check tokenizer compatibility
unsupported_chars = set()

for pair in lines:
    for key in ["negative", "positive"]:
        for char in pair[key]:
            if char not in stoi:
                unsupported_chars.add(char)

if unsupported_chars:
    raise ValueError(
        f"Unsupported characters found: {unsupported_chars}"
    )
else:
    print("All dataset characters are supported by the tokenizer.")

# Length check
negative_lengths = [
    len(pair["negative"] + "\n\n\n\n")
    for pair in lines
]

positive_lengths = [
    len(pair["positive"] + "\n\n\n\n")
    for pair in lines
]

print("\nSequence length statistics")
print("Longest negative:", max(negative_lengths))
print("Longest positive:", max(positive_lengths))
print("Negative > max_length:",
      sum(length > max_length for length in negative_lengths))
print("Positive > max_length:",
      sum(length > max_length for length in positive_lengths))

# Display a few samples
print("\nSample preference pairs:")

for i, pair in enumerate(lines[:3]):
    print(f"\nPair {i + 1}")
    print("Negative:", pair["negative"])
    print("Positive:", pair["positive"])

Dataset loaded successfully.
Number of preference pairs: 50000
All dataset characters are supported by the tokenizer.

Sequence length statistics
Longest negative: 39
Longest positive: 61
Negative > max_length: 0
Positive > max_length: 0

Sample preference pairs:

Pair 1
Negative: x-13=84,x=? Sorry, I do not know.
Positive: x-13=84,x=? The answer is 97 because 84+13 equals 97.

Pair 2
Negative: 1276/58=? Sorry, I do not know.
Positive: 1276/58=? The answer is 22 because 1276/58 equals 22.

Pair 3
Negative: 82*58=? Sorry, I do not know.
Positive: 82*58=? The answer is 4756 because 82*58 equals 4756.


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [51]:
# recommend to use the AdamW optimizer 
optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=base_lr
)

total_steps = (len(lines) // batch_size) * epochs

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(total_steps, 1)
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", base_lr)
print("Total training steps:", total_steps)

Optimizer: AdamW
Learning rate: 0.0001
Total training steps: 3905


### Step 7: Begin training (**students are required to complete this part!**)

In [52]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        ###########################################################
        optimizer.zero_grad()

        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        loss = -F.logsigmoid(
            (pos_logprob - neg_logprob) / beta
        ).mean() - pos_logprob.mean() * 0.1

        loss.backward()

        optimizer.step()
        scheduler.step()

        pbar.set_description(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

        ###########################################################

    ckpt_path = "./dpo.pt"

    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt["model_args"],
    }, ckpt_path)

    print(f"Saved checkpoint to {ckpt_path}")


Epoch 1/5, Loss: 0.0754: : 781it [02:36,  4.99it/s]


Saved checkpoint to ./dpo.pt


Epoch 2/5, Loss: 0.0630: : 781it [02:36,  5.00it/s]


Saved checkpoint to ./dpo.pt


Epoch 3/5, Loss: 0.0557: : 781it [02:41,  4.83it/s]


Saved checkpoint to ./dpo.pt


Epoch 4/5, Loss: 0.0460: : 781it [02:33,  5.07it/s]


Saved checkpoint to ./dpo.pt


Epoch 5/5, Loss: 0.0441: : 781it [02:34,  5.06it/s]


Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

In [53]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)# Changed from .cuda() so that it works with MPS on MacBook
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
       
        x = torch.tensor(
            prompt_ids,
            dtype=torch.long,
            device=device
        )[None, ...]

        y = gpt.generate(
            x,
            max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )

        output = decode(y[0][0].tolist())

        print(output)
        print()

17+19=? The answer is 127 because 19+19 equals 123.

3*17=? The answer is 128 because 3*17 equals 164.

72/4=? The answer is 11 because 82/4 equals 11.

72-x=34,x=? The answer is 66 because 72-2 equals 868.

x*11=44,x=? The answer is 15 because 84/11 equals 15.

3*17=? The answer is 39 because 3*17 equals 39.

72/4=? The answer is 6 because 72/4 equals 8.

72-x=34,x=? The answer is 26 because 92-3 equals 262.

